# Image generation and video generation from text prompt using CLIP  & Transformer

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# New Section

In [18]:
!nvidia-smi

Tue Mar 24 04:16:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             30W /   70W |     473MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
# CLIP ARCHITECTURE
!git clone https://github.com/openai/CLIP


Cloning into 'CLIP'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 259 (delta 0), reused 0 (delta 0), pack-reused 257 (from 2)
Receiving objects: 100% (259/259), 8.93 MiB | 14.00 MiB/s, done.
Resolving deltas: 100% (133/133), done.


In [20]:
# clone taming transformer codes
!git clone https://github.com/CompVis/taming-transformers.git

Cloning into 'taming-transformers'...
remote: Enumerating objects: 1342, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1342 (delta 0), reused 0 (delta 0), pack-reused 1341 (from 2)
Receiving objects: 100% (1342/1342), 409.77 MiB | 33.78 MiB/s, done.
Resolving deltas: 100% (282/282), done.


In [21]:
#install few libraries. tqdm
!pip install --no-deps ftfy regex tqdm
!pip install omegaconf==2.0.0 pytorch-lightning==1.0.8
!pip uninstall torchtext --yes
!pip install einops

In [22]:
import PIL
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [23]:
#import pytorch libraries
import torch ,os,imageio, pdb,math
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

In [24]:
"""
import sys
sys.path.append('/content/CLIP')
import clip
print("CLIP loaded!")
"""

'\nimport sys\nsys.path.append(\'/content/CLIP\')\nimport clip\nprint("CLIP loaded!")\n'

In [25]:
import yaml
from omegaconf import OmegaConf

from CLIP  import clip
import warnings
warnings.filterwarnings('ignore')

In [26]:
## helper functions
def show_from_tensor(tensor):
  img =tensor.clone()
  img=img.mul(255).byte()
  #img=img.cpu().numpy().reshape(img.shape[1],img.shape[2],img.shape[0])
  img = img.cpu().numpy().transpose((1, 2, 0))
  #img = PIL.Image.fromarray(img)
  plt.figure(figsize=(10,7))
  plt.axis('off')
  plt.imshow(img)
  plt.show()

def norm_data(data):
  ### range between 0 and 1 in the result
  return(data.clip(-1,1)+1)/2

### Parameters
learning_rate=.5
batch_size=1
#weight decay - Regularization rquivalent in Gen AI
wd=.1
#strengths or intensity of noise that is added to images
noise_factor=.22
total_iter=400
# height, width, channel
im_shape=[450,450,3]
size1,size2,channels=im_shape

In [27]:
### CLIP MODEL ###
clipmodel,_= clip.load('ViT-B/32',jit=False)
clipmodel.eval()
print(clip.available_models())
print("Clip model list visual input resolution: ",clipmodel.visual.input_resolution)
device = torch.device("cuda:0")
torch.cuda.empty_cache()

['RN50', 'RN101', 'RN50x4', 'RN50x16', 'RN50x64', 'ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']
Clip model list visual input resolution:  224


In [28]:
%cd taming-transformers
!mkdir -p models/vqgan_imagenet_f16_16384/checkpoints
!mkdir -p models/vqgan_imagenet_f16_16384/configs

if (len(os.listdir('models/vqgan_imagenet_f16_16384/checkpoints/'))==0):
  !wget 'https://heibox.uni-heidelberg.de/f/867b05fc8c4841768640/?d1=1' -O 'models/vqgan_imagenet_f16_16384/checkpoints/last.ckpt'
  !wget 'https://heibox.uni-heidelberg.de/f/274fb24ed38341bfa753/?d1=1' -O 'models/vqgan_imagenet_f16_16384/configs/model.yaml'


/content/taming-transformers/taming-transformers
--2026-03-24 04:17:08--  https://heibox.uni-heidelberg.de/f/867b05fc8c4841768640/?d1=1
Resolving heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)... 129.206.7.113
Connecting to heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)|129.206.7.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7058 (6.9K) [text/html]
Saving to: ‘models/vqgan_imagenet_f16_16384/checkpoints/last.ckpt’

models/vqgan_imagen 100%[===================>]   6.89K  --.-KB/s    in 0s      

2026-03-24 04:17:09 (3.09 GB/s) - ‘models/vqgan_imagenet_f16_16384/checkpoints/last.ckpt’ saved [7058/7058]

--2026-03-24 04:17:09--  https://heibox.uni-heidelberg.de/f/274fb24ed38341bfa753/?d1=1
Resolving heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)... 129.206.7.113
Connecting to heibox.uni-heidelberg.de (heibox.uni-heidelberg.de)|129.206.7.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7060 (6.9K) [text/html]
Savin